# ETL de BANCI: autónomo, local y paso a paso

Esta versión del procesamiento BANCI define sus funciones y parámetros en el notebook: **no importa `config.py` ni módulos `src/*.py`** y solo lee un archivo local. En cada etapa muestra dimensiones, tipos, nulos y una vista previa para ver cómo cambian los datos.

Las vistas enmascaran identificadores y datos personales. La exportación está desactivada hasta que revises el resultado. Ejecuta las celdas en orden y empieza por la configuración.

## 1. Preparación y configuración

Ajusta las rutas del Excel, la hoja, el filtro por entidad y las opciones de salida. El filtro se aplica al final, después de validar la entrega completa, para que los indicadores de calidad describan toda la base.

In [ ]:
# 1. CONFIGURACIÓN: todas las rutas y reglas editables viven en este cuaderno.
from pathlib import Path
from datetime import datetime
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Insumo local configurado en config.py a la fecha de creación de esta copia.
INPUT_BANCI = PROJECT_ROOT / "input" / "banci" / "SUBIR_R_202609_septiembre.xlsx"
SHEET_BANCI = "Victimas"  # también puede ser un índice, por ejemplo 0
OUTPUT_FILE = PROJECT_ROOT / "output" / "Resultado_2_Procesamiento_ETL_BANCI_autonomo.parquet"
EXPORTAR_RESULTADO = False
FORMATO_SALIDA = "parquet"  # "parquet" o "csv"
MOSTRAR_DATOS_SENSIBLES = False
ID_ESTADO_FILTRO = None  # ID INEGI opcional; None procesa todo el país
VALIDACION_ESTRICTA = True

# Las mismas columnas internas y alias que usa el ETL vigente.
COLUMN_MAPPING = {
    "nombres": ["nombres", "nombre"],
    "primer_apellido": ["primer_apellido", "paterno", "apellido_paterno"],
    "segundo_apellido": ["segundo_apellido", "materno", "apellido_materno"],
    "curp": ["curp", "curp_victima"],
    "sexo": ["sexo"],
    "fha_nac": ["fha_nac", "fecha_nacimiento", "fecha_nac"],
    "entidad": ["entidad", "entidad_federativa", "estado_reporta", "entidad_reporta"],
    "id_mun_hchos": ["id_mun_hchos", "idmunicipio_hechos", "id_municipio_hechos"],
    "nom_mun_hchos": ["nom_mun_hchos", "municipio_hechos", "nombre_municipio_hechos", "municipio"],
    "fha_de_hchos": ["fha_de_hchos", "fecha_hechos", "fecha de hechos"],
    "fha_de_ini": ["fha_de_ini", "fecha_inicio", "fecha inicio"],
    "localizado_o_no_localizado": ["localizado_o_no_localizado", "estatus", "estatus_localizado"],
    "fue_delito": ["fue_delito", "fue_delito_origen"],
    "delito_hom": ["delito_hom", "delito_homologado"],
    "llave": ["llave"],
    "folio_rnpdno": ["folio_rnpdno", "folio_rnpdno_ori", "fub_ok"],
    "ntra_ci": ["ntra_ci", "ntraci"],
    "id_ci": ["id_ci", "idci"],
}
COLUMNAS_REQUERIDAS = [
    "nombres", "primer_apellido", "segundo_apellido", "curp", "sexo", "fha_nac",
    "entidad", "id_mun_hchos", "fha_de_hchos", "fha_de_ini",
    "localizado_o_no_localizado", "llave", "folio_rnpdno",
]
COLUMNAS_OPCIONALES = ["fue_delito", "delito_hom", "nom_mun_hchos", "id_ci", "ntra_ci"]
CODIGOS_ESTADO_CURP = {
    "AS", "BC", "BS", "CC", "CL", "CM", "CS", "CH", "DF", "DG", "GT", "GR",
    "HG", "JC", "MC", "MN", "NT", "NL", "OC", "PL", "QT", "QR", "SP",
    "SL", "SR", "TC", "TS", "TL", "VZ", "YN", "ZS", "NE",
}
ENTIDADES = {
    1:"AGUASCALIENTES", 2:"BAJA CALIFORNIA", 3:"BAJA CALIFORNIA SUR", 4:"CAMPECHE",
    5:"COAHUILA DE ZARAGOZA", 6:"COLIMA", 7:"CHIAPAS", 8:"CHIHUAHUA", 9:"CIUDAD DE MEXICO",
    10:"DURANGO", 11:"GUANAJUATO", 12:"GUERRERO", 13:"HIDALGO", 14:"JALISCO",
    15:"ESTADO DE MEXICO", 16:"MICHOACAN DE OCAMPO", 17:"MORELOS", 18:"NAYARIT",
    19:"NUEVO LEON", 20:"OAXACA", 21:"PUEBLA", 22:"QUERETARO", 23:"QUINTANA ROO",
    24:"SAN LUIS POTOSI", 25:"SINALOA", 26:"SONORA", 27:"TABASCO", 28:"TAMAULIPAS",
    29:"TLAXCALA", 30:"VERACRUZ", 31:"YUCATAN", 32:"ZACATECAS",
}
AUDITORIA = []

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Insumo BANCI local: {INPUT_BANCI}")
print(f"Hoja: {SHEET_BANCI}")
print("No se importan módulos .py del proyecto ni se usa Drive.")

## 2. Funciones auxiliares y auditoría

Las funciones siguientes están definidas aquí mismo. La auditoría registra filas, columnas, memoria, nulos y tipos; las muestras son pequeñas y ocultan por defecto nombres, CURP, folios, llaves, expedientes, teléfonos y contactos.

In [ ]:
# 2. FUNCIONES: normalización, estandarización, CURP, fechas y snapshots.
def normalizar_encabezado(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(c for c in texto if not unicodedata.combining(c)).lower().strip()
    texto = re.sub(r"[\s.\-]+", "_", texto)
    return re.sub(r"[^a-z0-9_]", "", texto)


def normalizar_texto(serie):
    """Normaliza Series de texto a mayúsculas, sin acentos (conserva Ñ), y trim."""
    tabla = str.maketrans({"Á":"A", "É":"E", "Í":"I", "Ó":"O", "Ú":"U", "Ü":"U"})
    def uno(valor):
        if pd.isna(valor):
            return pd.NA
        return re.sub(r"\s+", " ", str(valor).upper().translate(tabla)).strip()
    return serie.map(uno) if isinstance(serie, pd.Series) else uno(serie)


def limpiar_curp(serie):
    """Replica la limpieza de CURP aplicada en el ETL BANCI original."""
    return (serie.astype(str).str.replace(r"\s+", "", regex=True).str.upper()
            .replace(["NAN", "NONE", ""], np.nan))


def validar_curp(serie):
    """Replica las cuatro reglas de es_curp_valida_vectorizado del ETL original."""
    curp = limpiar_curp(serie)
    longitud = curp.str.len() == 18
    formato = curp.str.contains(r"^[A-Z]{4}\d{6}[HM]", regex=True, na=False, flags=re.IGNORECASE)
    homoclave = curp.str.contains(r"[A-Z]{3}.{2}$", regex=True, na=False, flags=re.IGNORECASE)
    estado = curp.str.slice(11, 13).str.upper().isin(CODIGOS_ESTADO_CURP)
    return (longitud & formato & homoclave & estado).fillna(False).astype(bool)


def estandarizar_columnas(df, mapping):
    """Renombra aliases al nombre estándar usando encabezados normalizados."""
    df = df.copy()
    columnas_normalizadas = {normalizar_encabezado(c): c for c in df.columns}
    cambios = {}
    for destino, alias in mapping.items():
        if destino in df.columns:
            continue
        encontrados = [columnas_normalizadas[normalizar_encabezado(a)] for a in alias
                       if normalizar_encabezado(a) in columnas_normalizadas]
        if encontrados:
            cambios[encontrados[0]] = destino
    if cambios:
        print("Columnas renombradas:", cambios)
        df = df.rename(columns=cambios)
    return df


def convertir_fecha(serie):
    """Admite fechas comunes y números de serie de Excel; valores ilegibles pasan a NaT."""
    fechas = pd.to_datetime(serie, errors="coerce")
    numeros = pd.to_numeric(serie, errors="coerce")
    excel = pd.to_datetime(numeros, unit="D", origin="1899-12-30", errors="coerce")
    return fechas.where(fechas.notna(), excel.where(numeros.between(1, 100_000)))


def vista_segura(df, n=5):
    muestra = df.head(n).copy()
    if not MOSTRAR_DATOS_SENSIBLES:
        patron = re.compile(
            r"nombre|apellido|curp|folio|fub|llave|carpeta|ntra|id_ci|telefono|tel_|correo|domicilio|direccion",
            flags=re.IGNORECASE,
        )
        for col in muestra.columns:
            if patron.search(str(col)):
                muestra[col] = muestra[col].map(lambda v: "[OCULTO]" if pd.notna(v) else v)
    return muestra


def auditar(etapa, df, mostrar=True):
    nulos = df.isna().sum().sort_values(ascending=False)
    datos = {
        "etapa": etapa, "filas": len(df), "columnas": df.shape[1],
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        "nulos_totales": int(df.isna().sum().sum()),
        "columnas_con_nulos": int((nulos > 0).sum()),
    }
    AUDITORIA.append(datos)
    print(f"\n[{etapa}] {len(df):,} filas × {df.shape[1]} columnas | memoria {datos['memoria_mb']} MB")
    print("Tipos (primeras 15):", df.dtypes.astype(str).head(15).to_dict())
    if (nulos > 0).any():
        print("Más nulos:", nulos[nulos > 0].head(8).to_dict())
    if mostrar:
        display(vista_segura(df))
    return datos


def corregir_anio_futuro(df, columna, fecha_limite):
    """Resta años a fechas futuras hasta que no superen el corte, respetando 29-feb."""
    salida = df.copy()
    if columna not in salida:
        return salida
    fechas = convertir_fecha(salida[columna])
    limite = pd.Timestamp(fecha_limite)
    mascara = fechas > limite
    for indice in fechas.index[mascara]:
        fecha = fechas.at[indice]
        while fecha > limite:
            try:
                fecha = fecha.replace(year=fecha.year - 1)
            except ValueError:
                fecha = fecha.replace(month=2, day=28, year=fecha.year - 1)
        fechas.at[indice] = fecha
    salida[columna] = fechas
    return salida

print("Funciones auxiliares cargadas; reglas de CURP replicadas del ETL original.")

## 3. Carga del archivo local y primer snapshot

Se comprueba que existe el Excel y se lee la hoja configurada. El snapshot inicial representa los valores tal como pandas los interpretó; las funciones posteriores trabajan sobre copias para dejar claro el antes y el después.

In [ ]:
# 3. LECTURA LOCAL: no se descarga ni se accede a Drive.
if not INPUT_BANCI.exists():
    raise FileNotFoundError(f"No existe el Excel BANCI configurado: {INPUT_BANCI}")

# Dejar que pandas infiera fechas y números como en el flujo original; así las columnas
# *_banci preservan mejor los tipos de origen. Las llaves se convertirán a texto aparte.
banci_original = pd.read_excel(INPUT_BANCI, sheet_name=SHEET_BANCI)
banci_original = banci_original.replace(["", " ", "NA", "N/A", "NaN", "nan", "None"], pd.NA)
banci_entrada = banci_original.copy(deep=True)
print(f"Archivo: {INPUT_BANCI.name} | Hoja: {SHEET_BANCI}")
auditar("01 - lectura del Excel", banci_entrada)

## 4. Inspección inicial del DataFrame

Se revisan las columnas disponibles, la forma de la tabla, el porcentaje de faltantes, estatus, entidades y fechas. Las cifras ayudan a distinguir datos realmente ausentes de columnas que no suelen llegar en ciertos cortes.

In [ ]:
# 4. PERFIL DE LA TABLA ANTES DE LIMPIAR.
print("Encabezados originales:")
display(pd.DataFrame({"columna": banci_entrada.columns, "tipo": banci_entrada.dtypes.astype(str)}))
print("\nFilas y columnas:", banci_entrada.shape)
print("\nNulos por columna (primeras 25):")
display(banci_entrada.isna().sum().sort_values(ascending=False).head(25).to_frame("nulos"))
auditar("02 - inspección inicial", banci_entrada, mostrar=False)

## 5. Normalización y mapeo de columnas

Se estandarizan encabezados y alias sin modificar los valores. Después se guarda una copia de las columnas del Excel con sufijo `_banci`; estas columnas sirven como respaldo del origen, mientras que las columnas internas podrán limpiarse y transformarse.

In [ ]:
# 5. ESTANDARIZAR Y RESPALDAR LOS VALORES CRUDOS ANTES DE TRANSFORMAR.
df_banci = banci_entrada.copy()
# Limpieza inicial de encabezados, luego mapeo flexible a los nombres internos.
df_banci.columns = [normalizar_encabezado(c) for c in df_banci.columns]
if len(set(df_banci.columns)) != len(df_banci.columns):
    repetidos = pd.Index(df_banci.columns)[pd.Index(df_banci.columns).duplicated()].unique().tolist()
    raise ValueError(f"Encabezados duplicados tras normalizar: {repetidos}")
df_banci = estandarizar_columnas(df_banci, COLUMN_MAPPING)

# Conservar los valores como llegaron, tras la normalización/mapeo del nombre del campo.
for columna in list(df_banci.columns):
    respaldo = f"{columna}_banci"
    if not columna.endswith("_banci") and respaldo not in df_banci.columns:
        df_banci[respaldo] = df_banci[columna].copy()

faltan = [c for c in COLUMNAS_REQUERIDAS if c not in df_banci.columns]
faltan_opcionales = [c for c in COLUMNAS_OPCIONALES if c not in df_banci.columns]
if faltan:
    raise ValueError(f"Faltan columnas requeridas {faltan}. Encabezados disponibles: {list(df_banci.columns)}")
if faltan_opcionales:
    print(f"Aviso: faltan columnas opcionales; se crearán vacías si las requiere el esquema: {faltan_opcionales}")

print(f"Estandarización completa: {len(df_banci):,} filas; columnas crudas `_banci` añadidas.")
auditar("03 - columnas estandarizadas y respaldo crudo", df_banci)

## 6. Validaciones del contenido

Se comprueba que existan las columnas base, que la entrega tenga filas, que las fechas de inicio se puedan interpretar y que los estatus tengan categorías conocidas. Estas comprobaciones no importan la comparación histórica del módulo original: aquí mostramos corte, distribución y avisos del archivo actual para que la revisión sea visible y autocontenida.

In [ ]:
# 6. VALIDAR DATOS ORIGINALES COMPLETOS ANTES DE REPARAR O FILTRAR.
if df_banci.empty:
    raise ValueError("La hoja BANCI no contiene registros.")

fecha_inicio_original = convertir_fecha(df_banci["fha_de_ini"])
fecha_inicio_futura = fecha_inicio_original > pd.Timestamp.now().normalize()
fechas_inicio_validas = fecha_inicio_original.where(~fecha_inicio_futura)
FECHA_CORTE_BANCI = fechas_inicio_validas.max()
if pd.isna(FECHA_CORTE_BANCI):
    mensaje = "No hay valores interpretables en fha_de_ini; no se puede calcular el corte BANCI."
    if VALIDACION_ESTRICTA:
        raise ValueError(mensaje)
    print("Aviso:", mensaje)
else:
    print(f"Corte detectado desde fha_de_ini (ignorando futuras): {FECHA_CORTE_BANCI.date()}")

n_fechas_futuras = int(fecha_inicio_futura.sum())
if n_fechas_futuras:
    print(f"Aviso: {n_fechas_futuras:,} fechas de inicio posteriores a hoy; no se usan para calcular el corte.")

estatus_original = df_banci["localizado_o_no_localizado"].astype("string").str.strip().str.upper()
patron_estatus = r"^(NO )?LOCALIZAD[AO]S?$"
estatus_raros = estatus_original[~estatus_original.str.match(patron_estatus, na=False)]
if len(estatus_raros):
    mensaje = f"{len(estatus_raros):,} filas tienen estatus no reconocido: {estatus_raros.value_counts().head(8).to_dict()}"
    if VALIDACION_ESTRICTA:
        raise ValueError(mensaje)
    print("Aviso:", mensaje)

print("Distribución de estatus:")
display(estatus_original.value_counts(dropna=False).to_frame("registros"))
print("Distribución de entidades (primeras 20):")
display(df_banci["entidad"].astype("string").value_counts(dropna=False).head(20).to_frame("registros"))
auditar("04 - validación de insumo", df_banci, mostrar=False)

## 7. Reparación de nombres y creación de derivados

La reparación toma el campo `nombres` como posible nombre completo solo cuando existe texto y falta al menos un apellido. Conserva expresiones como “de la” o “del” como parte de un apellido compuesto. Luego se crean campos normalizados para los cruces y se mantienen intactas las copias `_banci`.

In [ ]:
# 7. REPARAR NOMBRES; después construir una fila por expediente con columnas de análisis.
def reparar_nombres(df):
    """Replica la división de nombres del ETL original cuando falta un apellido."""
    salida = df.copy()
    for columna in ["nombres", "primer_apellido", "segundo_apellido"]:
        if columna not in salida:
            salida[columna] = np.nan
    nombres = salida["nombres"].astype(object)
    falta_apellido = salida["primer_apellido"].isna() | salida["segundo_apellido"].isna()
    mascara = nombres.notna() & falta_apellido
    indices = salida.index[mascara]
    print(f"Registros que requieren revisar/dividir el nombre: {len(indices):,}")

    patron_preposicion = re.compile(r"\b(de\s+la|de\s+las|de\s+los|de|del|la|las|los|y)\s+", re.IGNORECASE)
    for indice in indices:
        nombre_original = str(salida.at[indice, "nombres"]) if pd.notna(salida.at[indice, "nombres"]) else ""
        ape1_original = str(salida.at[indice, "primer_apellido"]) if pd.notna(salida.at[indice, "primer_apellido"]) else ""
        ape2_original = str(salida.at[indice, "segundo_apellido"]) if pd.notna(salida.at[indice, "segundo_apellido"]) else ""
        if not (len(nombre_original) > 0 and (len(ape1_original) == 0 or len(ape2_original) == 0)):
            continue

        nombre_protegido = patron_preposicion.sub(lambda m: m.group(0).replace(" ", "_"), nombre_original.strip())
        partes = [p for p in re.sub(r"\s+", " ", nombre_protegido).split(" ") if p]
        n = len(partes)
        nuevo_nombre = nombre_original
        nuevo_ape1 = salida.at[indice, "primer_apellido"]
        nuevo_ape2 = salida.at[indice, "segundo_apellido"]
        if n >= 3:
            nuevo_nombre = " ".join(partes[:n-2])
            nuevo_ape1 = partes[n-2]
            nuevo_ape2 = partes[n-1]
        elif n == 2:
            nuevo_nombre = partes[0]
            nuevo_ape1 = partes[1]
            nuevo_ape2 = None
        else:
            # El código original deja el nombre intacto pero vacía ambos apellidos.
            nuevo_nombre = nombre_original
            nuevo_ape1 = None
            nuevo_ape2 = None

        def limpiar_parte(valor):
            return valor.replace("_", " ") if valor else None
        salida.at[indice, "nombres"] = limpiar_parte(nuevo_nombre)
        salida.at[indice, "primer_apellido"] = limpiar_parte(nuevo_ape1)
        salida.at[indice, "segundo_apellido"] = limpiar_parte(nuevo_ape2)
    return salida


def crear_variables_derivadas(df, fecha_corte):
    """Añade columnas transformadas para análisis y matching; preserva columnas origen."""
    salida = reparar_nombres(df)
    for columna in ["nombres", "primer_apellido", "segundo_apellido"]:
        texto = salida[columna].astype(str).replace(r"^0$", np.nan, regex=True).replace("nan", np.nan)
        salida[columna] = texto
        salida[f"{columna}_derivada_banci"] = texto.copy()

    nombre_crudo = (salida["nombres"].fillna("").astype(str) + " "
        + salida["primer_apellido"].fillna("").astype(str) + " "
        + salida["segundo_apellido"].fillna("").astype(str))
    salida["nombre_completo_derivada_banci"] = normalizar_texto(nombre_crudo)
    ruido = r"SE DESCONOCE|RESTOS OSEOS|CUERPO|SIN VIDA|DESCONOCIDO|NO IDENTIFICADO|\bSD\b|\bNA\b|^M$|([a-z]\.){2,}[a-z]?"
    nombre = salida["nombre_completo_derivada_banci"].fillna("").astype(str)
    salida["es_nombre_valido_derivada_banci"] = (
        salida["nombre_completo_derivada_banci"].notna()
        & ~nombre.str.contains(r"\d", regex=True)
        & ~nombre.str.contains(ruido, flags=re.IGNORECASE, regex=True)
        & nombre.str.len().ge(11)
    )
    salida["curp_valida_derivada_banci"] = validar_curp(limpiar_curp(salida["curp"]).astype(str)) if "curp" in salida else False

    salida["fha_de_hchos_derivada_banci"] = convertir_fecha(salida["fha_de_hchos"])
    salida = corregir_anio_futuro(salida, "fha_de_hchos_derivada_banci", fecha_corte or pd.Timestamp.now())
    salida["fha_nac_derivada_banci"] = convertir_fecha(salida["fha_nac"])

    for columna, derivada in [("id_mun_hchos", "id_mun_hchos_derivada_banci"),
                              ("llave", "llave_derivada_banci"),
                              ("folio_rnpdno", "fub_derivada_banci")]:
        origen = columna if columna in salida else f"{columna}_banci"
        valores = salida[origen] if origen in salida else pd.Series(np.nan, index=salida.index)
        valores = valores.astype(str).replace("nan", np.nan).replace("None", np.nan)
        salida[derivada] = valores
        salida[columna] = valores

    salida["localizado_o_no_localizado_derivada_banci"] = salida["localizado_o_no_localizado"].astype(str).str.strip().str.upper()
    salida["entidad_derivada_banci"] = normalizar_texto(salida["entidad"])
    for columna, derivada in [("ntra_ci", "ntra_ci_derivada_banci"), ("id_ci", "id_ci_derivada_banci")]:
        origen = columna if columna in salida else f"{columna}_banci"
        valores = salida[origen] if origen in salida else pd.Series(np.nan, index=salida.index)
        salida[derivada] = valores.astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "<NA>": np.nan, "": np.nan})

    salida["nom_mun_hchos_derivada_banci"] = normalizar_texto(salida["nom_mun_hchos"]) if "nom_mun_hchos" in salida else np.nan
    carpeta = salida["ntra_ci_derivada_banci"].fillna("").astype(str).str.strip()
    idci = salida["id_ci_derivada_banci"].fillna("").astype(str).str.strip()
    carpeta_real = ~(carpeta.eq("") | carpeta.str.fullmatch(r"0+", na=False))
    parte = normalizar_texto(carpeta.where(carpeta_real, idci))
    entidad = normalizar_texto(salida["entidad_derivada_banci"])
    salida["id_carpeta_derivada_banci"] = (entidad + "/" + parte).where(parte.str.strip() != "", np.nan)

    identidad_invalida = (~salida["curp_valida_derivada_banci"]) & (~salida["es_nombre_valido_derivada_banci"])
    fub = salida["fub_derivada_banci"]
    tiene_fub = fub.notna() & (fub != "") & (fub != "nan")
    mask_skip = identidad_invalida & (~tiene_fub)
    salida["skip_match_derivada_banci"] = False
    salida.loc[mask_skip, "skip_match_derivada_banci"] = True
    salida["comentario_match_derivada_banci"] = None
    salida.loc[mask_skip, "comentario_match_derivada_banci"] = "Identidad Inválida (Sin Nombre ni CURP válidos)"
    return salida


df_derivado = crear_variables_derivadas(df_banci, FECHA_CORTE_BANCI)
auditar("05 - reparación de nombres y variables derivadas", df_derivado)

## 8. Filtro opcional, esquema final y DataFrame maestro

Si configuras un ID de entidad, se filtran registros por nombre normalizado. Después seleccionamos el esquema fijo del archivo BANCI de salida. Si una columna opcional no venía en el Excel, se agrega como nula para conservar las mismas columnas entre cortes.

In [ ]:
# 8. APLICAR FILTRO OPCIONAL Y REDUCIR AL CONTRATO ESTABLE DE SALIDA.
COLUMNAS_SALIDA = [
    "entidad_banci", "llave_banci", "id_ci_banci", "ntra_ci_banci", "fha_de_ini_banci",
    "fha_de_hchos_banci", "nom_mun_hchos_banci", "id_mun_hchos_banci", "delito_hom_banci",
    "sexo_banci", "fha_nac_banci", "folio_rnpdno_banci", "primer_apellido_banci",
    "segundo_apellido_banci", "nombres_banci", "curp_banci",
    "localizado_o_no_localizado_banci", "fue_delito_banci", "fub_ok_banci",
    "entidad_derivada_banci", "id_ci_derivada_banci", "ntra_ci_derivada_banci",
    "fha_de_hchos_derivada_banci", "nom_mun_hchos_derivada_banci",
    "localizado_o_no_localizado_derivada_banci", "fha_nac_derivada_banci",
    "nombres_derivada_banci", "primer_apellido_derivada_banci",
    "segundo_apellido_derivada_banci", "nombre_completo_derivada_banci",
    "curp_valida_derivada_banci", "id_carpeta_derivada_banci",
]

if ID_ESTADO_FILTRO is not None:
    entidad_objetivo = normalizar_texto(pd.Series([ENTIDADES[ID_ESTADO_FILTRO]])).iloc[0]
    n_antes = len(df_derivado)
    df_derivado = df_derivado.loc[df_derivado["entidad_derivada_banci"].eq(entidad_objetivo)].copy()
    print(f"Filtro de entidad {entidad_objetivo}: {n_antes:,} → {len(df_derivado):,} filas")
else:
    print("Sin filtro estatal: se conserva el universo nacional.")

# La columna histórica fub_ok_banci puede venir como alias que se estandarizó a folio_rnpdno.
if "fub_ok_banci" not in df_derivado:
    if "folio_rnpdno_banci" in df_derivado:
        df_derivado["fub_ok_banci"] = df_derivado["folio_rnpdno_banci"].copy()
    else:
        df_derivado["fub_ok_banci"] = pd.NA

# Columnas opcionales que falten se crean vacías; nunca improvisar datos.
for columna in COLUMNAS_SALIDA:
    if columna not in df_derivado:
        df_derivado[columna] = pd.NA

df_maestro = df_derivado[COLUMNAS_SALIDA].copy()
# Homogeneizar strings/NA para que el resultado pueda guardarse en Parquet sin tipos mixtos.
for columna in df_maestro.columns:
    if pd.api.types.is_object_dtype(df_maestro[columna].dtype):
        df_maestro[columna] = df_maestro[columna].astype("string").replace({"nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})

auditar("06 - esquema final BANCI", df_maestro)

## 9. Revisión final, trazabilidad y exportación

El cierre compara el número de filas de entrada y salida, muestra las banderas de calidad y el corte detectado. La tabla `AUDITORIA` permite revisar la evolución del DataFrame sin guardar copias completas de cada paso. La escritura del archivo es opt-in.

In [ ]:
# 9. MÉTRICAS DEL RESULTADO Y EXPORTACIÓN SOLO SI SE ACTIVA EXPLÍCITAMENTE.
print("Evolución de filas:")
print(f"  Entrada Excel:       {len(banci_entrada):,}")
print(f"  Tras estandarizar:   {len(df_banci):,}")
print(f"  Tras derivados:      {len(df_derivado):,}")
print(f"  Maestro de salida:   {len(df_maestro):,} filas × {df_maestro.shape[1]} columnas")
print(f"  Corte de FHA_DE_INI: {FECHA_CORTE_BANCI}")
print(f"  Fechas de hechos aún futuras al corte: {int((df_derivado['fha_de_hchos_derivada_banci'] > FECHA_CORTE_BANCI).sum()) if pd.notna(FECHA_CORTE_BANCI) else 'sin corte'}")
print(f"  CURP válidas: {int(df_derivado['curp_valida_derivada_banci'].sum()):,}")
print(f"  Nombres válidos: {int(df_derivado['es_nombre_valido_derivada_banci'].sum()):,}")
print(f"  Registros omitidos del matching: {int(df_derivado['skip_match_derivada_banci'].sum()):,}")
print("\nDistribución final por estatus:")
display(df_maestro["localizado_o_no_localizado_derivada_banci"].value_counts(dropna=False).to_frame("registros"))
print("\nEtapas registradas:")
display(pd.DataFrame(AUDITORIA))
print("\nVista segura del maestro final:")
display(vista_segura(df_maestro))

if EXPORTAR_RESULTADO:
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    if FORMATO_SALIDA.lower() == "parquet":
        df_maestro.to_parquet(OUTPUT_FILE, index=False, engine="pyarrow")
        archivo_salida = OUTPUT_FILE
    elif FORMATO_SALIDA.lower() == "csv":
        archivo_salida = OUTPUT_FILE.with_suffix(".csv")
        df_maestro.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
    else:
        raise ValueError("FORMATO_SALIDA debe ser 'parquet' o 'csv'.")
    print(f"Exportado: {archivo_salida}")
else:
    print("Exportación apagada. Activa EXPORTAR_RESULTADO en la configuración tras revisar el DataFrame.")

## 10. Comparación opcional con el Parquet del ETL original

Si ya existe `output/Resultado_2_Procesamiento_ETL_BANCI.parquet`, esta celda compara el maestro actual contra esa salida: columnas, filas y valores alineados por posición. Solo muestra conteos de diferencias, no datos personales. Asegúrate de que ambos archivos correspondan al mismo Excel/corte para interpretar el resultado.

In [ ]:
# 10. COMPARAR CON LA ÚLTIMA SALIDA PERSISTIDA DEL ETL ORIGINAL, SIN MODIFICARLA.
ruta_original = PROJECT_ROOT / "output" / "Resultado_2_Procesamiento_ETL_BANCI.parquet"
if not ruta_original.exists():
    print(f"No hay Parquet de referencia para comparar: {ruta_original}")
else:
    df_original_etl = pd.read_parquet(ruta_original)
    cols_actual = list(df_maestro.columns)
    cols_original = list(df_original_etl.columns)
    cols_comunes = [c for c in cols_actual if c in df_original_etl.columns]
    print(f"Parquet de referencia: {ruta_original.name}")
    print(f"Tamaño referencia: {df_original_etl.shape} | autónomo: {df_maestro.shape}")
    print(f"Columnas solo en referencia: {sorted(set(cols_original) - set(cols_actual))}")
    print(f"Columnas solo en autónomo: {sorted(set(cols_actual) - set(cols_original))}")
    diffs = {}
    if len(df_original_etl) == len(df_maestro):
        for columna in cols_comunes:
            izq = df_original_etl[columna].astype("string").fillna("<NA>").reset_index(drop=True)
            der = df_maestro[columna].astype("string").fillna("<NA>").reset_index(drop=True)
            distintos = izq.ne(der)
            if distintos.any():
                diffs[columna] = int(distintos.sum())
        print(f"Columnas comunes con diferencias por posición: {len(diffs)} de {len(cols_comunes)}")
        if diffs:
            display(pd.Series(diffs, name="celdas_distintas").sort_values(ascending=False).to_frame())
        else:
            print("Los valores de las columnas comunes coinciden fila por fila.")
        if "curp_valida_derivada_banci" in diffs:
            print("\nCruce de banderas CURP (referencia × autónomo), sin revelar CURP:")
            display(pd.crosstab(
                df_original_etl["curp_valida_derivada_banci"].astype("string").fillna("<NA>"),
                df_maestro["curp_valida_derivada_banci"].astype("string").fillna("<NA>"),
                rownames=["original"], colnames=["autónomo"], dropna=False,
            ))
    else:
        print("No se compara valor por valor: las cantidades de filas son distintas.")
    mismas_columnas_y_orden = cols_original == cols_actual
    misma_cantidad = len(df_original_etl) == len(df_maestro)
    iguales = mismas_columnas_y_orden and misma_cantidad and not diffs
    print(f"¿Salidas iguales en esquema, orden y valores comparados? {iguales}")